
# Feature Engineering
Tujuan: Menyiapkan dataset hasil *preprocessing* menjadi dataset yang siap digunakan pada tahap pemodelan (*model-ready dataset*). Pada tahap ini akan dilakukan pemilihan fitur, pembuatan variabel target, serta penentuan strategi pembagian data untuk menghindari *data leakage*.

## 1. Loading
Memuat dataset hasil *preprocessing* yang telah dibersihkan pada tahap sebelumnya. Dataset ini akan menjadi dasar dalam proses *feature engineering*.

In [1]:
import pandas as pd

In [2]:
results = pd.read_csv("../data/interim/results_clean.csv")

### Preview Dataset

In [3]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


Dataset hasil preprocessing berhasil dimuat dan siap digunakan pada proses feature engineering.

## 2. Feature Selection
Memutuskan dan memilih fitur yang akan digunakan.

| Feature      | Status | Digunakan Sebagai | Alasan                                                                       |
| ------------ | :----: | ----------------- | ---------------------------------------------------------------------------- |
| `home_team`  |    ✅   | Feature           | Identitas tim kandang sebelum pertandingan berlangsung.                      |
| `away_team`  |    ✅   | Feature           | Identitas tim tandang sebelum pertandingan berlangsung.                      |
| `tournament` |    ✅   | Feature           | Mewakili konteks dan tingkat kompetisi pertandingan.                         |
| `neutral`    |    ✅   | Feature           | Berkaitan dengan fenomena *Home Advantage*.                                  |
| `date`       |   ⚙️   | Referensi         | Digunakan untuk *time-based split*, bukan sebagai fitur baseline.            |
| `city`       |    ❌   | Tidak digunakan   | Jumlah kategori sangat banyak dan belum dievaluasi manfaatnya pada baseline. |
| `country`    |    ❌   | Tidak digunakan   | Belum digunakan untuk menjaga kesederhanaan model awal.                      |
| `home_score` |    ❌   | Tidak digunakan   | Menyebabkan *target leakage*.                                                |
| `away_score` |    ❌   | Tidak digunakan   | Menyebabkan *target leakage*.                                                |


### Insight
Tabel diatas berisi keputusan mengenai penanganan fitur-fitur yang ada.

## 3. Target Engineering
Tujuan utama model adalah memprediksi "pemenang", sehingga kita akan membuat kolom baru untuk target yaitu `match_result`.

In [4]:
def get_match_result(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    if row["home_score"] < row["away_score"]:
        return "A"
    else:
        return "D"
    
results["match_result"] = results.apply(get_match_result, axis=1)

In [5]:
# VALIDASI
results[
    ["home_score",
     "away_score",
     "match_result"]
].head(10)

,home_score,away_score,match_result
0,0.0,0.0,D
1,4.0,2.0,H
2,2.0,1.0,H
3,2.0,2.0,D
4,3.0,0.0,H
5,4.0,0.0,H
6,1.0,3.0,A
7,0.0,2.0,A
8,7.0,2.0,H
9,9.0,0.0,H


In [6]:
results["match_result"].value_counts()

match_result
H    24227
A    13963
D    11243
Name: count, dtype: int64

### Insight:
Validasi data sudah menunjukkan kesesuaian kolom hasil pertandingan dengan skor kandang dan tandang. Variabel target match_result berhasil dibuat berdasarkan perbandingan skor kandang (home_score) dan skor tandang (away_score). Variabel ini memiliki tiga kelas, yaitu H (Home Win), D (Draw), dan A (Away Win), yang akan digunakan sebagai target pada model klasifikasi. 

Selain itu, jumlah Home Win, Away Win, dan Draw sudah sesuai dengan catatan analisis pada Exploratory Data Analysis bagian 4.1 yang membahas distribusi hasil pertandingan.

## 4. Feature Transformation
Ubah bentuk, skala, atau distribusi data agar lebih mudah dipahami dan diproses oleh algoritma machine learning

### 4.1 Feature Transformation Strategy
Seluruh fitur kategorikal (home_team, away_team, dan tournament) belum ditransformasikan menjadi bentuk numerik pada notebook ini. Proses encoding akan dilakukan menggunakan Pipeline pada tahap pemodelan sehingga encoder hanya mempelajari data latih (training data). Pendekatan ini membantu menjaga alur eksperimen tetap rapi serta mengurangi risiko data leakage.

### 4.2 Feature Overview

In [7]:
results[
    [
        "home_team",
        "away_team",
        "tournament",
        "neutral",
        "match_result"
    ]
].head()

,home_team,away_team,tournament,neutral,match_result
0,Scotland,England,Friendly,False,D
1,England,Scotland,Friendly,False,H
2,Scotland,England,Friendly,False,H
3,England,Scotland,Friendly,False,D
4,Scotland,England,Friendly,False,H


Feature yang dipilih tetap dipertahankan dalam bentuk aslinya. Transformasi menjadi representasi numerik akan dilakukan pada notebook pemodelan menggunakan Pipeline sehingga proses pelatihan model menjadi lebih modular dan mudah direproduksi.

## 5. Save Feature Dataset


Kita akan menyimpan dataset baseline.

In [8]:
baseline_features = results[
    [
        "date",
        "home_team",
        "away_team",
        "tournament",
        "neutral",
        "match_result",
    ]
]

In [9]:
baseline_features.to_csv(
    "../data/processed/baseline_features.csv",
    index=False
)

In [10]:
baseline_features.head()

,date,home_team,away_team,tournament,neutral,match_result
0,1872-11-30,Scotland,England,Friendly,False,D
1,1873-03-08,England,Scotland,Friendly,False,H
2,1874-03-07,Scotland,England,Friendly,False,H
3,1875-03-06,England,Scotland,Friendly,False,D
4,1876-03-04,Scotland,England,Friendly,False,H


In [11]:
baseline_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   date          49433 non-null  str  
 1   home_team     49433 non-null  str  
 2   away_team     49433 non-null  str  
 3   tournament    49433 non-null  str  
 4   neutral       49433 non-null  bool 
 5   match_result  49433 non-null  str  
dtypes: bool(1), str(5)
memory usage: 1.9 MB


Baseline features berhasil disimpan.

## Final Insight
Tahap Feature Engineering berhasil menghasilkan dataset baseline yang hanya berisi fitur-fitur yang tersedia sebelum pertandingan dimulai, yaitu home_team, away_team, tournament, dan neutral, serta variabel target match_result. Seluruh fitur kategorikal masih dipertahankan dalam bentuk aslinya dan akan ditransformasikan melalui Pipeline pada tahap pemodelan. Pendekatan ini menjaga alur eksperimen tetap modular serta mengurangi risiko data leakage.

Dataset baseline yang telah disiapkan pada notebook ini akan digunakan pada tahap Baseline Modeling. Pada notebook berikutnya akan dilakukan pembagian data berbasis waktu (time-based split), pembangunan Pipeline untuk transformasi fitur, pelatihan model dasar seperti Logistic Regression dan Random Forest, serta evaluasi performa model.